# 第 14 章 · SGEMM ②：寄存器分块、向量化与 warptiling

本章回答：为什么"一个线程算 C 的 64 个元素"比"一个线程算 1 个"快 5 倍？外积（outer product）的写法是什么？向量化读、转置 As、调参、warptiling 各贡献了多少？从 13% 到 94% 的每一步在优化什么？

配套讲义：仓库里 `cuda-programming/notes/lessons/14-sgemm-register-tiling.md`（讲解、动画、练习都在讲义里）。对应原文：Boehm worklog Kernel 4–10（`materials/text/blog-boehm-cuda-matmul-worklog.txt` 第 383–806 行）；Gordić《Inside NVIDIA GPUs》warp-tiling 部分（`text/blog-gordic-inside-nvidia-gpus-matmul.txt`）；CUTLASS 的分层 GEMM 概念（`code/cutlass/cpp-docs/`）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二、三步 读 vs 乘加 | 读 $T_M + T_N$，乘加 $T_M T_N$ | `reads_and_fmas(tm, tn)` | 1×1：0.5；8×1：0.89；8×8：4 |
| 第三步 Kernel 5 配置 | 256 × 64 = 128 × 128 | `256 * 64 == 128 * 128` | 16384 |
| 第三步 外积 | `threadResults[m][n] += regM[m] * regN[n]` | `block_tile(As, Bs)`（每个线程 `thread_tile_outer`） | 和 `As @ Bs` 一致 |
| 第六步 全景 | ÷ cuBLAS | `BOEHM` | 1.3% → 93.7% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `BOEHM`**（来自 `ch14_register_tiling.py`）

In [ ]:
BOEHM = [("1", "naive", 309), ("2", "合并访存", 1987), ("3", "shared 分块", 2980), ("4", "1D 寄存器分块", 8475),
         ("5", "2D 寄存器分块（外积）", 15972), ("6", "向量化 + 转置 As", 18237), ("9", "自动调参", 19721),
         ("10", "warptiling", 21779)]

**常量 `CUBLAS`**（来自 `ch14_register_tiling.py`）

In [ ]:
CUBLAS = 23250

**函数 `thread_tile_outer`**（来自 `ch14_register_tiling.py`）：Kernel 5 的内循环：对每个 dotIdx，读 A 的一小列到 regM、B 的一小行到 regN，做 TM × TN 次乘加。

In [ ]:
def thread_tile_outer(As: np.ndarray, Bs: np.ndarray, row: int, col: int, tm: int = 8, tn: int = 8) -> np.ndarray:
    """Kernel 5 的内循环：对每个 dotIdx，读 A 的一小列到 regM、B 的一小行到 regN，做 TM × TN 次乘加。

    As: BM × BK（shared 里的 A tile），Bs: BK × BN；(row, col) 是线程在 block 里的位置。
    """
    acc = np.zeros((tm, tn), dtype=As.dtype)                # threadResults，放寄存器
    for dot in range(As.shape[1]):
        reg_m = As[row * tm:(row + 1) * tm, dot]            # 从 shared 读 TM 个
        reg_n = Bs[dot, col * tn:(col + 1) * tn]            # 从 shared 读 TN 个
        acc += np.outer(reg_m, reg_n)                       # TM × TN 次乘加
    return acc

**函数 `block_tile`**（来自 `ch14_register_tiling.py`）：block 里所有线程各算自己的小块，拼起来就是整个 BM × BN 的结果。

In [ ]:
def block_tile(As: np.ndarray, Bs: np.ndarray, tm: int = 8, tn: int = 8) -> np.ndarray:
    """block 里所有线程各算自己的小块，拼起来就是整个 BM × BN 的结果。"""
    bm, bn = As.shape[0], Bs.shape[1]
    out = np.zeros((bm, bn), dtype=As.dtype)
    for r in range(bm // tm):
        for c in range(bn // tn):
            out[r * tm:(r + 1) * tm, c * tn:(c + 1) * tn] = thread_tile_outer(As, Bs, r, c, tm, tn)
    return out

**函数 `reads_and_fmas`**（来自 `ch14_register_tiling.py`）：每个线程算 TM × TN 个结果：每步读 shared TM + TN 次（1 × 1 时是 2 次），做 TM × TN 次乘加。

In [ ]:
def reads_and_fmas(tm: int, tn: int) -> tuple[int, int, float]:
    """每个线程算 TM × TN 个结果：每步读 shared TM + TN 次（1 × 1 时是 2 次），做 TM × TN 次乘加。"""
    reads = tm + tn
    fmas = tm * tn
    return reads, fmas, fmas / reads

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二、三步 每步读 shared 几次、乘加几次")
for tm, tn in ((1, 1), (8, 1), (8, 8)):
    r, f, ratio = reads_and_fmas(tm, tn)
    print(f"  {tm} × {tn}: 读 {r:2d}，乘加 {f:2d}，乘加 ÷ 读 = {ratio:.2f}")

In [ ]:
print("\n第三步 Kernel 5 的配置：BM = BN = 128、BK = 8、256 个线程、每线程 8 × 8")
print("  256 × 64 =", 256 * 64, "= 128 × 128 =", 128 * 128)
rng = np.random.default_rng(0)
As = rng.standard_normal((128, 8)).astype(np.float32)
Bs = rng.standard_normal((8, 128)).astype(np.float32)
out = block_tile(As, Bs)
print("  按外积一步步算出的 128 × 128 和 As @ Bs 一致：", np.allclose(out, As @ Bs, atol=1e-5))
print("  线程 (0, 0) 的小块左上角：", out[0, 0], "≈", (As @ Bs)[0, 0])

In [ ]:
print("\n第六步 Boehm 全景（A6000，相对 cuBLAS 23250 GFLOP/s）")
for k, what, g in BOEHM:
    print(f"  Kernel {k:>2} {what:<14} {g:6d} GFLOP/s  {g / CUBLAS:6.1%}")

## 3. 代码里才看得到的

1. **`np.outer` 就是外积**：一列 × 一行得到一个小方块，正是 Kernel 5 内循环那两层 `for m / for n`。
2. **1 × 1 时读 2 次而不是 1 + 1 次**：公式 $T_M + T_N$ 在 1 × 1 时正好也是 2，和 Kernel 3 一样。
3. **`block_tile` 把 256 个线程排成 16 × 16**：每个线程 (r, c) 负责结果的第 r 块行、第 c 块列；拼起来就是整个 128 × 128。

## 4. 自己改一改

- 把 `block_tile(As, Bs)` 的 tm、tn 换成 8 × 4、4 × 8，确认结果仍和 `As @ Bs` 一致。
- 把 `As` 的 BK 从 8 改成 16（Kernel 9 在 A6000 上的最优值），确认结果不变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 每一层都分块：显存到 shared 一次，shared 到寄存器一次；
> 一人算一格读两次，一人算一方块读一圈：读的是边，算的是面积；
> 外积：读一列 A、一行 B，乘出一个小方块；
> 读宽一点（float4、LDS.128），参数自动调，换卡就重调；
> 中间加一层 warp tile，正好接上 Tensor Core。